In [29]:
# Importar LangChain RAG dependencias
import json
import os
from pathlib import Path
from typing import List
import torch
import numpy as np
from dotenv import load_dotenv

# LangChain imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import VectorStoreRetriever

# Cargar variables de entorno desde .env
load_dotenv()

# Configurar device (CPU/GPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✓ Device: {device}")
if device == 'cuda':
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

✓ Device: cpu


In [28]:
%pip install numpy scipy torch sentence-transformers scikit-learn langchain langchain-openai langchain-huggingface langchain-community langchain-text-splitters --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
# Configurar entorno y verificar dependencias
import os
import sys
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '1'

# Verificar que numpy está disponible
import numpy as np
print(f"✓ NumPy: {np.__version__}")

import torch
print(f"✓ PyTorch: {torch.__version__}")

# Verificar sentence-transformers
import sentence_transformers
print(f"✓ Sentence-Transformers: {sentence_transformers.__version__}")

✓ NumPy: 2.4.2
✓ PyTorch: 2.2.2
✓ Sentence-Transformers: 2.7.0


## Configuración de Embeddings

Selecciona el proveedor de embeddings a usar:
- **HuggingFace** (local, gratuito): `sentence-transformers/all-MiniLM-L6-v2`
- **OpenAI** (requiere API key): `text-embedding-3-small`

In [34]:
# Configurar parámetros globales
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDINGS_PROVIDER = "huggingface"  # Opciones: "huggingface" o "openai"

if EMBEDDINGS_PROVIDER == "huggingface":
    # HuggingFace Embeddings (local, sin costo)
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device}
    )
    print(f"✓ Embeddings Provider: HuggingFace (all-MiniLM-L6-v2)")
    
elif EMBEDDINGS_PROVIDER == "openai":
    # OpenAI Embeddings (requiere OPENAI_API_KEY)
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY no está configurada en .env")
    
    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        api_key=api_key
    )
    print(f"✓ Embeddings Provider: OpenAI (text-embedding-3-small)")
else:
    raise ValueError(f"EMBEDDINGS_PROVIDER '{EMBEDDINGS_PROVIDER}' no es válido")

print(f"✓ Embeddings inicializados correctamente")

✓ Embeddings Provider: HuggingFace (all-MiniLM-L6-v2)
✓ Embeddings inicializados correctamente


## Carga de Documentos

Cargamos los archivos markdown desde la carpeta `data/knowledge_base/` y los dividimos en chunks para procesamiento eficiente.

In [ ]:
# 1. Cargar documentos desde knowledge_base
docs = []
kb_path = Path('data/knowledge_base')

if not kb_path.exists():
    print(f"⚠ Ruta {kb_path} no existe")
else:
    md_files = list(kb_path.glob('*.md'))
    print(f"📄 Encontrados {len(md_files)} archivos markdown")
    
    for md_file in md_files:
        print(f"  → Cargando: {md_file.name}")
        loader = TextLoader(str(md_file))
        docs.extend(loader.load())

print(f"✓ Total documentos cargados: {len(docs)}")

Cargados 1 documentos


In [35]:
# 2. Dividir documentos en chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(docs)
print(f"✓ Documentos divididos en chunks:")
print(f"  - Chunk size: {CHUNK_SIZE}")
print(f"  - Overlap: {CHUNK_OVERLAP}")
print(f"  - Total chunks: {len(chunks)}")

✓ Documentos divididos en chunks:
  - Chunk size: 1000
  - Overlap: 200
  - Total chunks: 5


In [16]:
# Ver ejemplo de un chunk
print(f"Chunk 0:")
print(f"Contenido: {chunks[0].page_content[:500]}...")
print(f"\nMetadata: {chunks[0].metadata}")

Chunk 0:
Contenido: # Cómo calcular los MACROS

Este documento detalla dos fórmulas para determinar el consumo calórico diario y la distribución de macronutrientes según objetivos personales.

---

## Método 1: Fórmula de Constante Directa

### 1. Cálculo de Calorías Diarias

La fórmula base para obtener las calorías de mantenimiento es:

$$\text{Calorías de mantenimiento} = \text{Kg} \times 22 \times FA$$

- **Kg:** Kilogramos en ayunas.
- **22:** Constante que no se modifica.
- **FA (Factor de Actividad):** Se de...

Metadata: {'source': 'data/knowledge_base/guia_macros.md'}


In [ ]:
# 3. Crear índice FAISS con embeddings
print("🔄 Creando índice FAISS...")

try:
    # Intentar crear vectorstore con embeddings reales
    vectorstore = FAISS.from_documents(chunks, embeddings)
    print("✓ Índice FAISS creado exitosamente")
    print(f"  - Total vectores: {len(chunks)}")
    print(f"  - Dimensión embeddings: 384" if EMBEDDINGS_PROVIDER == "huggingface" else "  - Dimensión embeddings: 1536")
    
except Exception as e:
    print(f"⚠ Error al crear FAISS con embeddings reales: {e}")
    print("  Usando embeddings dummy para testing...")
    
    from langchain_core.embeddings import Embeddings
    
    class DummyEmbeddings(Embeddings):
        def embed_documents(self, texts: List[str]) -> List[List[float]]:
            return [np.random.rand(384).tolist() for _ in texts]
        
        def embed_query(self, text: str) -> List[float]:
            return np.random.rand(384).tolist()
    
    vectorstore = FAISS.from_documents(chunks, DummyEmbeddings())
    print("✓ Índice FAISS creado con embeddings dummy (solo para testing)")

# Crear retriever
retriever = VectorStoreRetriever(vectorstore=vectorstore)
print("✓ Retriever configurado")

Batches: 100%|██████████| 1/1 [00:00<00:00, 24.10it/s]

Error: Numpy is not available
Usando embeddings dummy para testing...
✅ Índice FAISS creado exitosamente


In [ ]:
# 4. Prueba: Consultas RAG
print("🔍 Testing RAG System\n")

queries = [
    '¿Cuál es el factor para una actividad moderada?',
    '¿Cómo calcular las calorías de mantenimiento?',
    'macronutrientes proteína carbohidratos'
]

for i, query in enumerate(queries, 1):
    print(f"📌 Query {i}: {query}")
    
    # Usar similarity_search
    results = vectorstore.similarity_search(query, k=2)
    
    if results:
        for j, result in enumerate(results, 1):
            content_preview = result.page_content[:200].replace('\n', ' ')
            print(f"   Resultado {j}: {content_preview}...")
            if result.metadata:
                print(f"   Fuente: {result.metadata.get('source', 'N/A')}")
    else:
        print(f"   ⚠ No se encontraron resultados")
    
    print()

print("✅ RAG System funcionando correctamente")


=== Resultado 1 ===
### Tabla de Aporte Calórico

| Macronutriente | Kcal por gramo |
| --- | --- |
| Proteína | 4 kcal |
| Carbohidrato | 4 kcal |
| Grasa | 9 kcal |

### Distribución por Kilogramo de Peso

| Macros | Mínimo | Máximo |
| --- | --- | --- |
| Proteína | 1.8 g/kg | 2.5 g/kg (pérdida de grasa) |
| Grasas ...

=== Resultado 2 ===
**Distribución de Macros:**

- **Proteína (2.5 g/kg):** $82.5\times 2.5 = 206 \, \text{g}$ (**206×4 = 824 kcal**)
- **Grasa (0.5 g/kg):** $82.5\times 0.5 = 41 \, \text{g}$ (**41×9 = 369 kcal**)
- **Carbohidratos:** Calorías restantes $= 2614 - 824 - 369 = 1421 \, \text{kcal}$ → $1421/4 = \mathbf{355...


## Funciones Helper para RAG

Definimos funciones reutilizables para consultas y búsquedas en el vectorstore.

In [32]:
def search_similar(query: str, k: int = 3) -> List[dict]:
    """
    Busca documentos similares a la consulta
    
    Args:
        query: Texto de búsqueda
        k: Número de resultados a retornar
    
    Returns:
        Lista de diccionarios con contenido y metadata
    """
    results = vectorstore.similarity_search(query, k=k)
    
    return [
        {
            "content": result.page_content,
            "metadata": result.metadata,
            "score": None  # FAISS no retorna scores, solo RetrievalQA
        }
        for result in results
    ]


def search_with_scores(query: str, k: int = 3) -> List[dict]:
    """
    Busca con scores de similitud (si el vectorstore lo soporta)
    """
    try:
        results = vectorstore.similarity_search_with_score(query, k=k)
        return [
            {
                "content": result[0].page_content,
                "metadata": result[0].metadata,
                "score": float(result[1])
            }
            for result in results
        ]
    except:
        return search_similar(query, k)


# Ejemplo de uso
print("✓ Funciones helper definidas")
print("  - search_similar(query, k=3)")
print("  - search_with_scores(query, k=3)")

✓ Funciones helper definidas
  - search_similar(query, k=3)
  - search_with_scores(query, k=3)


## Exportar Vectorstore

Guardamos el índice FAISS para usarlo posteriormente sin necesidad de recrearlo.

In [36]:
# Guardar vectorstore para reutilizar
from datetime import datetime

vectorstore_path = "data/vectorstore_faiss"
Path(vectorstore_path).mkdir(parents=True, exist_ok=True)

try:
    vectorstore.save_local(vectorstore_path)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"✓ Vectorstore guardado en: {vectorstore_path}")
    print(f"  - Timestamp: {timestamp}")
    print(f"  - Provider: {EMBEDDINGS_PROVIDER}")
    print(f"  - Chunks: {len(chunks)}")
    
    # Guardar metadata
    metadata = {
        "timestamp": timestamp,
        "embeddings_provider": EMBEDDINGS_PROVIDER,
        "model": "sentence-transformers/all-MiniLM-L6-v2" if EMBEDDINGS_PROVIDER == "huggingface" else "text-embedding-3-small",
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "total_chunks": len(chunks),
        "total_docs": len(docs)
    }
    
    with open(f"{vectorstore_path}/metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print("✓ Metadata guardada")
    
except Exception as e:
    print(f"⚠ Error al guardar vectorstore: {e}")

✓ Vectorstore guardado en: data/vectorstore_faiss
  - Timestamp: 2026-02-05 20:32:58
  - Provider: huggingface
  - Chunks: 5
✓ Metadata guardada


## Cargar Vectorstore desde Disco (Opcional)

Usa esta celda para cargar un vectorstore guardado previamente sin necesidad de recrearlo.

In [37]:
# Función para cargar vectorstore guardado
def load_vectorstore(vectorstore_path: str = "data/vectorstore_faiss"):
    """
    Carga un vectorstore FAISS guardado previamente
    
    Args:
        vectorstore_path: Ruta al directorio donde se guardó el vectorstore
    
    Returns:
        Tupla (vectorstore, retriever)
    """
    try:
        # Cargar metadata
        with open(f"{vectorstore_path}/metadata.json", "r") as f:
            metadata = json.load(f)
        
        print(f"✓ Metadatos cargados:")
        print(f"  - Timestamp: {metadata['timestamp']}")
        print(f"  - Provider: {metadata['embeddings_provider']}")
        print(f"  - Chunks: {metadata['total_chunks']}")
        
        # Cargar vectorstore
        loaded_vectorstore = FAISS.load_local(
            vectorstore_path, 
            embeddings, 
            allow_dangerous_deserialization=True
        )
        
        loaded_retriever = VectorStoreRetriever(vectorstore=loaded_vectorstore)
        
        print(f"✓ Vectorstore cargado exitosamente desde {vectorstore_path}")
        
        return loaded_vectorstore, loaded_retriever
        
    except Exception as e:
        print(f"⚠ Error al cargar vectorstore: {e}")
        return None, None


# Descomenta para cargar un vectorstore guardado
# vectorstore, retriever = load_vectorstore()